# Structural Damage - Entrainement complet

Notebook dedie a un entrainement robuste de segmentation multi-classes:
- DenseNet201 encodeur + tete de segmentation,
- loss combinee CE ponderee + Dice,
- AMP, clipping gradient, OneCycleLR,
- early stopping, checkpoints `last`/`best`, reprise auto.

In [1]:
from pathlib import Path
import os, json, time, random
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.models as tvm
from torchvision.models import DenseNet201_Weights
import torchvision.transforms.functional as TF

NOTEBOOK_DIR = Path('.').resolve()
DATA_ROOT = NOTEBOOK_DIR.parent / 'data' / 'cracks_dataset'
TRAIN_ROOT = DATA_ROOT / 'ready_for_training' / 'train'
VAL_ROOT = DATA_ROOT / 'ready_for_training' / 'val'
TRAIN_CSV = TRAIN_ROOT / 'train_labels.csv'
VAL_CSV = VAL_ROOT / 'val_labels.csv'

assert VAL_CSV.exists(), f'CSV val introuvable: {VAL_CSV}'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

N_CLASSES = 8
IMG_SIZE = 320

TRAIN_CFG = {
    'seed': 42,
    'img_size': IMG_SIZE,
    'batch_size': 6,
    'epochs': 30,
    'freeze_epochs': 5,
    'lr_head': 2e-3,
    'lr_full': 5e-4,
    'weight_decay': 1e-4,
    'grad_clip': 1.0,
    'num_workers': 4,
    'persistent_workers': True,
    'pin_memory': torch.cuda.is_available(),
    'max_train_samples': None,
    'max_val_samples': None,
    'class_hist_sample': 1024,
    'early_stopping_patience': 8,
    'dice_weight': 0.6,
    'ce_weight': 0.4,
    'resume_last': True,
    'save_dir': str(NOTEBOOK_DIR.parent / 'models' / 'struct_damage_full'),
}

print('Device:', DEVICE)
print(json.dumps(TRAIN_CFG, indent=2, ensure_ascii=False))

Device: cpu
{
  "seed": 42,
  "img_size": 320,
  "batch_size": 6,
  "epochs": 30,
  "freeze_epochs": 5,
  "lr_head": 0.002,
  "lr_full": 0.0005,
  "weight_decay": 0.0001,
  "grad_clip": 1.0,
  "num_workers": 4,
  "persistent_workers": true,
  "pin_memory": false,
  "max_train_samples": null,
  "max_val_samples": null,
  "class_hist_sample": 1024,
  "early_stopping_patience": 8,
  "dice_weight": 0.6,
  "ce_weight": 0.4,
  "resume_last": true,
  "save_dir": "C:\\Users\\mvm\\open3d_vision\\models\\struct_damage_full"
}


In [2]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True

set_seed(TRAIN_CFG['seed'])

if TRAIN_CSV.exists() and VAL_CSV.exists():
    df_tr = pd.read_csv(TRAIN_CSV)
    df_va = pd.read_csv(VAL_CSV)
    root_tr, root_va = TRAIN_ROOT, VAL_ROOT
else:
    df_all = pd.read_csv(VAL_CSV).sample(frac=1.0, random_state=TRAIN_CFG['seed']).reset_index(drop=True)
    cut = int(0.8 * len(df_all))
    df_tr, df_va = df_all.iloc[:cut].copy(), df_all.iloc[cut:].copy()
    root_tr = root_va = VAL_ROOT

if TRAIN_CFG['max_train_samples'] is not None:
    df_tr = df_tr.head(int(TRAIN_CFG['max_train_samples']))
if TRAIN_CFG['max_val_samples'] is not None:
    df_va = df_va.head(int(TRAIN_CFG['max_val_samples']))

print(f'Train: {len(df_tr):,} | Val: {len(df_va):,}')

HAS_ALBU = True
try:
    import albumentations as A
except Exception:
    HAS_ALBU = False
    print('albumentations absent -> fallback torchvision')

Train: 42,000 | Val: 10,500
albumentations absent -> fallback torchvision


In [3]:
class FullSegDataset(Dataset):
    MEAN = [0.485, 0.456, 0.406]
    STD  = [0.229, 0.224, 0.225]

    def __init__(self, df, root: Path, img_size=320, train=False):
        self.df = df.reset_index(drop=True)
        self.root = Path(root)
        self.size = int(img_size)
        self.train = bool(train)
        if self.train and HAS_ALBU:
            self.albu = A.Compose([
                A.RandomResizedCrop(size=(self.size, self.size), scale=(0.7, 1.0), ratio=(0.75, 1.33), p=0.7),
                A.HorizontalFlip(p=0.5),
                A.VerticalFlip(p=0.2),
                A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.10, rotate_limit=15, border_mode=0, p=0.5),
                A.RandomBrightnessContrast(0.2, 0.2, p=0.4),
                A.GaussianBlur(blur_limit=(3, 5), p=0.15),
            ])
        else:
            self.albu = None

    def __len__(self):
        return len(self.df)

    def _fallback_aug(self, image, mask):
        if random.random() < 0.5:
            image, mask = TF.hflip(image), TF.hflip(mask)
        if random.random() < 0.2:
            image, mask = TF.vflip(image), TF.vflip(mask)
        image = TF.adjust_brightness(image, 0.8 + 0.4 * random.random())
        image = TF.adjust_contrast(image, 0.8 + 0.4 * random.random())
        return image, mask

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(self.root / row['image_path']).convert('RGB')
        mask = Image.open(self.root / row['mask_path'])

        if self.albu is not None:
            out = self.albu(image=np.array(image), mask=np.array(mask, dtype=np.int64))
            image = Image.fromarray(out['image'])
            mask = Image.fromarray(out['mask'].astype(np.uint8))
        else:
            image = image.resize((self.size, self.size), Image.BILINEAR)
            mask = mask.resize((self.size, self.size), Image.NEAREST)
            if self.train:
                image, mask = self._fallback_aug(image, mask)

        image_t = TF.to_tensor(image)
        image_t = TF.normalize(image_t, self.MEAN, self.STD)
        mask_t = torch.from_numpy(np.array(mask, dtype=np.int64)).clamp(0, N_CLASSES - 1)
        return image_t, mask_t

train_ds = FullSegDataset(df_tr, root_tr, img_size=TRAIN_CFG['img_size'], train=True)
val_ds = FullSegDataset(df_va, root_va, img_size=TRAIN_CFG['img_size'], train=False)

nw = int(TRAIN_CFG['num_workers'])
pw = bool(TRAIN_CFG['persistent_workers']) and nw > 0
train_dl = DataLoader(train_ds, batch_size=int(TRAIN_CFG['batch_size']), shuffle=True, num_workers=nw, pin_memory=bool(TRAIN_CFG['pin_memory']), persistent_workers=pw)
val_dl = DataLoader(val_ds, batch_size=int(TRAIN_CFG['batch_size']), shuffle=False, num_workers=nw, pin_memory=bool(TRAIN_CFG['pin_memory']), persistent_workers=pw)

print(f'train batches={len(train_dl)} | val batches={len(val_dl)}')

train batches=7000 | val batches=1750


In [4]:
class DenseNet201Seg(nn.Module):
    def __init__(self, num_classes=8, pretrained=True):
        super().__init__()
        weights = DenseNet201_Weights.IMAGENET1K_V1 if pretrained else None
        backbone = tvm.densenet201(weights=weights)
        self.encoder = backbone.features
        self.decoder = nn.Sequential(
            nn.Conv2d(1920, 256, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.Dropout2d(0.2),
            nn.Conv2d(256, num_classes, kernel_size=1),
        )

    def forward(self, x):
        feat = F.relu(self.encoder(x), inplace=True)
        logits = self.decoder(feat)
        return F.interpolate(logits, size=x.shape[2:], mode='bilinear', align_corners=False)

    def freeze_encoder(self, freeze=True):
        for p in self.encoder.parameters():
            p.requires_grad = not freeze

hist_n = min(len(df_tr), int(TRAIN_CFG['class_hist_sample']))
sample_idx = np.random.choice(len(df_tr), size=hist_n, replace=False) if hist_n > 0 else []
class_hist = np.zeros(N_CLASSES, dtype=np.float64)
for i in sample_idx:
    mp = root_tr / df_tr.iloc[int(i)]['mask_path']
    arr = np.array(Image.open(mp), dtype=np.int64)
    arr = np.clip(arr, 0, N_CLASSES - 1)
    class_hist += np.bincount(arr.reshape(-1), minlength=N_CLASSES)

freq = class_hist / max(class_hist.sum(), 1.0)
weights_np = 1.0 / np.log(1.02 + freq + 1e-8)
weights_np = weights_np / weights_np.mean()
weights_np[0] *= 0.35
class_weights = torch.tensor(weights_np, dtype=torch.float32, device=DEVICE)
print('class_weights:', np.round(weights_np, 3).tolist())

class DiceCELoss(nn.Module):
    def __init__(self, ce_weight=None, dice_weight=0.6, ce_mix=0.4, eps=1e-6):
        super().__init__()
        self.ce = nn.CrossEntropyLoss(weight=ce_weight)
        self.dw = float(dice_weight)
        self.cw = float(ce_mix)
        self.eps = eps
    def forward(self, logits, target):
        ce = self.ce(logits, target)
        probs = torch.softmax(logits, dim=1)
        onehot = F.one_hot(target, num_classes=logits.shape[1]).permute(0, 3, 1, 2).float()
        inter = (probs * onehot).sum((0, 2, 3))
        den = probs.sum((0, 2, 3)) + onehot.sum((0, 2, 3))
        dice = (2 * inter + self.eps) / (den + self.eps)
        return self.cw * ce + self.dw * (1.0 - dice.mean())

class_weights: [0.012, 1.069, 1.252, 1.058, 1.252, 1.237, 1.252, 0.844]


In [ ]:
model = DenseNet201Seg(num_classes=N_CLASSES, pretrained=True).to(DEVICE)
model.freeze_encoder(True)
criterion = DiceCELoss(class_weights, TRAIN_CFG['dice_weight'], TRAIN_CFG['ce_weight']).to(DEVICE)

optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=float(TRAIN_CFG['lr_head']), weight_decay=float(TRAIN_CFG['weight_decay']))
scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=float(TRAIN_CFG['lr_head']), epochs=int(TRAIN_CFG['epochs']), steps_per_epoch=max(1, len(train_dl)), pct_start=0.2)
scaler = torch.amp.GradScaler('cuda', enabled=(DEVICE.type == 'cuda'))

def train_epoch(model, loader, optimizer, criterion, scheduler, scaler, device, grad_clip=1.0):
    model.train(); total=0.0
    for imgs, masks in loader:
        imgs, masks = imgs.to(device, non_blocking=True), masks.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=device.type, enabled=(device.type == 'cuda')):
            logits = model(imgs)
            loss = criterion(logits, masks)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=float(grad_clip))
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        total += loss.item() * imgs.size(0)
    return total / len(loader.dataset)

@torch.no_grad()
def eval_epoch(model, loader, criterion, device, nc):
    model.eval(); total=0.0
    inter = np.zeros(nc, dtype=np.float64)
    union = np.zeros(nc, dtype=np.float64)
    for imgs, masks in loader:
        imgs, masks = imgs.to(device, non_blocking=True), masks.to(device, non_blocking=True)
        logits = model(imgs)
        total += criterion(logits, masks).item() * imgs.size(0)
        preds = logits.argmax(1)
        for c in range(nc):
            pc, tc = (preds == c), (masks == c)
            inter[c] += (pc & tc).sum().item()
            union[c] += (pc | tc).sum().item()
    iou_cls = inter / (union + 1e-6)
    valid = union > 0
    miou = float(iou_cls[valid].mean()) if valid.any() else 0.0
    return total / len(loader.dataset), iou_cls, miou

save_dir = Path(TRAIN_CFG['save_dir']); save_dir.mkdir(parents=True, exist_ok=True)
ckpt_last = save_dir / 'densenet201_full_last.pth'
ckpt_best = save_dir / 'densenet201_full_best.pth'
hist_csv = save_dir / 'history_full.csv'

start_epoch = 1
best_miou = -1.0
patience_count = 0
history = []

if TRAIN_CFG['resume_last'] and ckpt_last.exists():
    ckpt = torch.load(ckpt_last, map_location=DEVICE)
    model.load_state_dict(ckpt['model_state_dict'])
    optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    scheduler.load_state_dict(ckpt['scheduler_state_dict'])
    scaler.load_state_dict(ckpt['scaler_state_dict'])
    start_epoch = int(ckpt['epoch']) + 1
    best_miou = float(ckpt.get('best_miou', -1.0))
    patience_count = int(ckpt.get('patience_count', 0))
    history = ckpt.get('history', [])
    print(f'Reprise depuis epoch {start_epoch} | best_miou={best_miou:.4f}')

epochs = int(TRAIN_CFG['epochs'])
freeze_epochs = int(TRAIN_CFG['freeze_epochs'])
patience = int(TRAIN_CFG['early_stopping_patience'])

print(f"{'Ep':>4} | {'TrainLoss':>9} | {'ValLoss':>8} | {'mIoU':>7} | {'LR':>9} | mode")
print('-' * 70)

for epoch in range(start_epoch, epochs + 1):
    if epoch == freeze_epochs + 1 and next(model.encoder.parameters()).requires_grad is False:
        model.freeze_encoder(False)
        optimizer = torch.optim.AdamW(model.parameters(), lr=float(TRAIN_CFG['lr_full']), weight_decay=float(TRAIN_CFG['weight_decay']))
        scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=float(TRAIN_CFG['lr_full']), epochs=max(1, epochs - freeze_epochs), steps_per_epoch=max(1, len(train_dl)), pct_start=0.2)
        print(f"-> Phase 2: encodeur degele, LR={TRAIN_CFG['lr_full']:.1e}")

    tr_loss = train_epoch(model, train_dl, optimizer, criterion, scheduler, scaler, DEVICE, grad_clip=TRAIN_CFG['grad_clip'])
    va_loss, iou_cls, va_miou = eval_epoch(model, val_dl, criterion, DEVICE, N_CLASSES)
    lr_now = optimizer.param_groups[0]['lr']
    mode = 'gele' if not next(model.encoder.parameters()).requires_grad else 'libre'
    print(f"{epoch:4d} | {tr_loss:9.4f} | {va_loss:8.4f} | {va_miou:7.4f} | {lr_now:9.2e} | {mode}")

    row = {'epoch': epoch, 'train_loss': tr_loss, 'val_loss': va_loss, 'val_miou': va_miou, 'lr': lr_now, 'mode': mode}
    for c in range(N_CLASSES):
        row[f'iou_{c}'] = float(iou_cls[c])
    history.append(row)

    improved = va_miou > best_miou
    if improved:
        best_miou = va_miou
        patience_count = 0
        torch.save({'epoch': epoch, 'best_miou': best_miou, 'model_state_dict': model.state_dict(), 'optimizer_state_dict': optimizer.state_dict(), 'scheduler_state_dict': scheduler.state_dict(), 'scaler_state_dict': scaler.state_dict(), 'train_cfg': TRAIN_CFG, 'history': history}, ckpt_best)
    else:
        patience_count += 1

    torch.save({'epoch': epoch, 'best_miou': best_miou, 'patience_count': patience_count, 'model_state_dict': model.state_dict(), 'optimizer_state_dict': optimizer.state_dict(), 'scheduler_state_dict': scheduler.state_dict(), 'scaler_state_dict': scaler.state_dict(), 'train_cfg': TRAIN_CFG, 'history': history}, ckpt_last)

    if patience_count >= patience:
        print(f'Early stopping: patience={patience} atteinte')
        break

pd.DataFrame(history).to_csv(hist_csv, index=False)
print('Termine')
print('best:', ckpt_best)
print('last:', ckpt_last)
print('history:', hist_csv)

  Ep | TrainLoss |  ValLoss |    mIoU |        LR | mode
----------------------------------------------------------------------
